# CitizenTech "Cidade Para Todos" — Detecção de buracos e rachaduras com YOLO11 · **v3.1 (resiliente)**
**Autor:** Octavio Augusto Ramalho Brandão Pires — **RM 555696** · Disciplina 09 — Visão Computacional · GS FIAP 2026.2

**O que mudou na v3.1 (corrige o problema de "notebook grande demais / sessão caiu"):**
- 🔁 **Treino resiliente:** os checkpoints vão **a cada época para o Google Drive**; se a sessão cair, a célula **Retomar** continua de onde parou (sem recomeçar).
- 🧹 **Saída leve:** barras de progresso são filtradas (antes o notebook chegava a 3 MB só de texto) e imagens são reduzidas antes de aparecer.
- ⏱️ **Treino mais curto:** 40 épocas (na sua execução anterior a validação já estava estável nas épocas 63–75: mAP@50 ≈ 0,56 e mAP@50-95 ≈ 0,30–0,31).
- 🚀 O treino roda **em segundo plano** (sobrevive a fechar a aba, enquanto a máquina virtual estiver ativa) e uma célula **monitora** o progresso.

**Fluxo:** Parte A (preparar dados) → Parte B (treinar) → Parte C (avaliar, demonstrar, entregar). **Se a sessão cair:** rode as células 1–3 e a Parte A de novo, depois a célula **Retomar** (Parte B).
Antes: `Ambiente de execução > Alterar tipo > GPU T4` · No Drive, em `GS_CV/`: **zip do projeto v3.1.0** e zip do dataset brasileiro.

In [ ]:
# 1) Ambiente + utilitários (rode de novo sempre que reconectar)
!pip -q install ultralytics huggingface_hub gradio seaborn roboflow
import os, sys, time, subprocess, shutil, json, glob, random, io
import torch
GPU = torch.cuda.is_available()
DEVICE = '0' if GPU else 'cpu'
print('GPU:', torch.cuda.get_device_name(0) if GPU else 'NÃO disponível -> ative a GPU T4 (senão o treino fica muito lento)')

def sh(cmd, every=30):
    """Executa o comando mostrando a saída, mas FILTRA barras de progresso (uma linha de status a cada `every` s).
    Levanta erro se o comando falhar."""
    print('$', cmd)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    buf, last = b'', 0.0
    while True:
        chunk = p.stdout.read1(4096)
        if not chunk and p.poll() is not None: break
        buf += chunk
        while b'\n' in buf:
            line, buf = buf.split(b'\n', 1)
            text = line.decode('utf-8', 'ignore').split('\r')[-1].strip()
            if text: print(text)
        if b'\r' in buf and time.time() - last > every:      # progresso (tqdm): mostra só o último estado, de tempos em tempos
            print('  …', buf.decode('utf-8', 'ignore').split('\r')[-2 if buf.endswith(b'\r') else -1].strip()[:140]); last = time.time(); buf = b''
    if buf.strip(): print(buf.decode('utf-8', 'ignore').split('\r')[-1].strip())
    if p.returncode: raise RuntimeError(f'FALHOU (código {p.returncode}): {cmd}')

def show(path, width=1000, q=72):
    """Mostra uma imagem REDUZIDA (JPEG) para não inflar o notebook."""
    from IPython.display import Image, display
    import cv2
    if not os.path.exists(path): print('(ainda não existe:', path, ')'); return
    im = cv2.imread(path)
    if im.shape[1] > width: im = cv2.resize(im, (width, int(im.shape[0] * width / im.shape[1])))
    display(Image(data=cv2.imencode('.jpg', im, [cv2.IMWRITE_JPEG_QUALITY, q])[1].tobytes(), format='jpeg'))

In [ ]:
# 2) Drive: arquivos de entrada, pasta de corridas (checkpoints) e backup
USE_DRIVE = True
PROJECT_ZIP = '/content/drive/MyDrive/GS_CV/cv_manutencao_urbana.zip'
DATASET_ZIP = '/content/drive/MyDrive/GS_CV/t576ydh9v8-3.zip'
DRIVE_RUNS  = '/content/drive/MyDrive/GS_CV/runs'        # checkpoints do treino (last.pt a cada época)
BACKUP_DIR  = '/content/drive/MyDrive/GS_CV/resultados'  # best.pt, curvas e resultados finais
if USE_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
else:
    from google.colab import files
    for n in files.upload():
        if n.startswith('cv_manutencao'): PROJECT_ZIP = '/content/' + n
        else: DATASET_ZIP = '/content/' + n
assert os.path.exists(PROJECT_ZIP), f'Não achei {PROJECT_ZIP}'
assert os.path.exists(DATASET_ZIP), f'Não achei {DATASET_ZIP}'
os.makedirs(DRIVE_RUNS, exist_ok=True); os.makedirs(BACKUP_DIR, exist_ok=True)
print('ok')

In [ ]:
# 3) Projeto + VERIFICAÇÃO DE VERSÃO (evita rodar zip antigo)
EXPECTED = '3.1.0'
sh('rm -rf /content/cv_manutencao_urbana')
sh(f'unzip -q -o "{PROJECT_ZIP}" -d /content')
os.chdir('/content/cv_manutencao_urbana')
ver = subprocess.run([sys.executable, '-c', 'from common import VERSION; print(VERSION)'], capture_output=True, text=True).stdout.strip()
if ver != EXPECTED:
    raise RuntimeError(f'ZIP DESATUALIZADO: projeto v{ver or "antiga"}, esperado v{EXPECTED}. Substitua o zip no Drive pelo mais novo.')
shutil.copy(DATASET_ZIP, 'dataset/')
sh('python run.py check'); print('projeto v' + ver + ' OK')

# PARTE A — Dados

## 4) Dataset brasileiro: formatos, máscaras → caixas YOLO, split por grupos de trechos

In [ ]:
sh('python run.py prep')

In [ ]:
import pandas as pd
r = json.load(open('reports/split_report.json'))
print(pd.DataFrame(r['splits']).T)
print('grupos compartilhados entre splits:', r['grupos_em_mais_de_um_split'], '| quadros descartados (zona-tampão):', r['imagens_descartadas_zona_tampao'])
print('auditoria de quase-duplicatas:', r.get('auditoria_quase_duplicatas'))
import cv2
fs = sorted(glob.glob('reports/label_check/*.jpg'))[:4]
tiles = [cv2.resize(cv2.imread(f), (512, 320)) for f in fs]
import numpy as np
cv2.imwrite('/tmp/label_check.jpg', np.vstack([np.hstack(tiles[:2]), np.hstack(tiles[2:4])]))
show('/tmp/label_check.jpg', 1000)   # conferência visual das caixas YOLO convertidas das máscaras

## 5) Fontes extras: RDD2022, buracos Roboflow e o SEU projeto Roboflow
RDD2022 (Hugging Face, CC BY-SA 4.0; sem drone; grupos por país) · buracos Roboflow (GitHub; só train/valid) · seu projeto Roboflow (chave nos **Secrets** do Colab: 🔑 → `ROBOFLOW_API_KEY`; **nunca no código**). O download do RDD2022 pode pausar alguns minutos por limite de requisições do Hugging Face.

In [ ]:
GROUPS_PER_COUNTRY = 8
for cmd in (f'python data_extra.py rdd2022 --groups_per_country {GROUPS_PER_COUNTRY}', 'python data_extra.py pothole_github'):
    try: sh(cmd)
    except RuntimeError as e: print('[aviso] fonte extra falhou, segue sem ela:', e)

In [ ]:
RF_ENABLED  = True
RF_WS, RF_PROJECT, RF_VERSION, RF_ALIAS = 'trabaio', 'analise-de-buracos-na-rua-8xzn9-z99wx-hlqzu-kiaft', 1, 'rf_rua'
if RF_ENABLED:
    key = None
    try:
        from google.colab import userdata; key = userdata.get('ROBOFLOW_API_KEY')
    except Exception: pass
    if not key:
        import getpass; key = getpass.getpass('Chave da API do Roboflow (vazio = pular): ').strip()
    if key:
        os.environ['ROBOFLOW_API_KEY'] = key
        try: sh(f'python data_extra.py roboflow --workspace {RF_WS} --project {RF_PROJECT} --version {RF_VERSION} --alias {RF_ALIAS}')
        except RuntimeError as e: print('[aviso] Roboflow falhou, segue sem ele:', e)
    else: print('Roboflow pulado (sem chave).')

In [ ]:
sh('python build_mix.py')
r = json.load(open('reports/mix_report.json'))
print(pd.DataFrame({f: {s: v['imagens'] for s, v in d.items()} for f, d in r['fontes'].items()}).T)
print('grupos compartilhados entre splits:', r['grupos_em_mais_de_um_split'])

**(Opcional)** Teste de fumaça: roda TODAS as etapas com treino minúsculo (poucos minutos) e limpa. Já passou na sua execução anterior; deixe `SMOKE=False` para economizar tempo.

In [ ]:
SMOKE = False
if SMOKE:
    for c in (f'python run.py train --smoke --device {DEVICE} --preset robust', f'python run.py eval --smoke --device {DEVICE}',
              'python run.py gallery', 'python run.py demo', 'python run.py sequence', 'python run.py report', 'python run.py clean'):
        sh(c)
    print('✅ smoke OK')

# PARTE B — Treino resiliente
Os checkpoints ficam no **Drive** (`GS_CV/runs/train/weights/last.pt`, atualizado a cada época). O treino roda em **segundo plano**; a célula de monitoramento mostra uma linha de status por minuto.
**Quanto tempo?** ~2 a 3 min por época na T4 (≈ 338 iterações/época com este dataset) → **40 épocas ≈ 1h30 a 2h**. Para uma versão mais rápida use `FAST=True` (512 px, 30 épocas, ≈ 1h).

In [ ]:
EPOCHS, FAST, COMPARE = 40, False, False
flags = '--epochs 30 --imgsz 512' if FAST else f'--epochs {EPOCHS}'
flags += ' --compare' if COMPARE else ''
cmd = f'python run.py train {flags} --device {DEVICE} --preset robust --project {DRIVE_RUNS} --backup_dir {BACKUP_DIR}'
subprocess.Popen(f'nohup {cmd} > train.log 2>&1 &', shell=True)
print('treino iniciado em segundo plano:', cmd)

In [ ]:
def status(csv):
    import pandas as pd
    if not os.path.exists(csv): return 'aguardando a 1ª época…'
    d = pd.read_csv(csv); d.columns = [c.strip() for c in d.columns]
    r = d.iloc[-1]
    return (f"época {int(r['epoch'])} | mAP@50 {r['metrics/mAP50(B)']:.3f} | mAP@50-95 {r['metrics/mAP50-95(B)']:.3f} | "
            f"P {r['metrics/precision(B)']:.3f} R {r['metrics/recall(B)']:.3f} | tempo acumulado {r['time']/60:.0f} min")

def monitorar(max_min=240, cada=60, name='train'):
    from IPython.display import clear_output
    csv = f'{DRIVE_RUNS}/{name}/results.csv'; t0 = time.time()
    while time.time() - t0 < max_min * 60:
        vivo = subprocess.run("pgrep -f train_model.py", shell=True, capture_output=True).returncode == 0
        clear_output(wait=True); print(time.strftime('%H:%M:%S'), '|', status(csv), '| treino ativo:', vivo)
        if not vivo: break
        time.sleep(cada)
    print('\n--- fim do log ---'); print(open('train.log', errors='ignore').read().replace('\r', '\n')[-700:])

monitorar()    # pode interromper e rodar de novo a qualquer momento; o treino continua em segundo plano

### 🔁 Retomar (use se a sessão/VM caiu no meio do treino)
1. Rode as células 1, 2, 3 e a **Parte A** de novo (o dataset em `/content` é refeito de forma idêntica: sementes fixas).  
2. Rode a célula abaixo: ela continua de `GS_CV/runs/train/weights/last.pt`, a partir da última época concluída.

In [ ]:
RESUME = False       # True para retomar
if RESUME:
    assert os.path.exists(f'{DRIVE_RUNS}/train/weights/last.pt'), 'last.pt não encontrado no Drive'
    cmd = f'python train_model.py --resume --name train --project {DRIVE_RUNS} --backup_dir {BACKUP_DIR} --yolo_dir data/yolo_mix'
    subprocess.Popen(f'nohup {cmd} > train.log 2>&1 &', shell=True); print('retomado:', cmd)
    monitorar()

In [ ]:
# Restaura os pesos treinados do Drive para o projeto (necessário após reconectar)
os.makedirs('weights', exist_ok=True)
for f in glob.glob(f'{BACKUP_DIR}/best*.pt'): shutil.copy(f, 'weights/')
if os.path.exists('weights/best_train.pt'): shutil.copy('weights/best_train.pt', 'weights/best.pt')
assert os.path.exists('weights/best.pt'), 'weights/best.pt ausente: o treino terminou?'
shutil.copy(f'{BACKUP_DIR}/results.csv', 'results_treino.csv') if os.path.exists(f'{BACKUP_DIR}/results.csv') else None
show(f'{BACKUP_DIR}/training_curves.png', 1000)
print('pesos prontos:', os.path.getsize('weights/best.pt') // 1024, 'KB')

# PARTE C — Avaliação, demonstrações e entrega

## 6) Avaliação no TESTE: métricas, matriz de confusão, por fonte, latência T4 × CPU, limiar e erros

In [ ]:
BENCH = 'cpu,0' if GPU else 'cpu'
sh(f'python evaluate.py --external_pothole --device {DEVICE} --bench_devices {BENCH}')
for f in ('reports/confusion_matrix.png', 'reports/f1_vs_conf.png', 'runs/eval_test/BoxPR_curve.png', 'runs/eval_test/BoxF1_curve.png'):
    show(f, 700)

## 7) 📊 Demonstrações visuais (geradas pelo modelo treinado)

In [ ]:
sh('python demo_gallery.py')
show('reports/demo/dashboard.png', 1100)
for f in sorted(glob.glob('reports/demo/teste_*.jpg')) + sorted(glob.glob('reports/demo/externas_*.jpg')):
    print(os.path.basename(f)); show(f, 1000)

## 8) Fotos externas independentes e sequência de imagens

In [ ]:
sh('python infer.py --source tests/external_images --out results/external --expected_from_folders')
d = pd.read_csv('results/external/expected_vs_detected.csv')
print(d.groupby('pasta').resultado_ok.agg(acertos='sum', imagens='count'))
sh('python sequence_demo.py --n_frames 40')
show('results/sequence/contact_sheet.jpg', 1000); show('results/sequence/timeline.png', 700)
print(json.dumps(json.load(open('results/sequence/sequence_summary.json')), indent=2, ensure_ascii=False))

## 9) 📷 Suas fotos (limiar escolhido e limiar sensível 0,10)

In [ ]:
from google.colab import files
import cv2, numpy as np, matplotlib.pyplot as plt
from cv_service import predict_urban_damage
up = files.upload()
for name, data in up.items():
    open(os.path.join('minhas_fotos', name), 'wb').write(data)
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
    ax[0].imshow(cv2.cvtColor(cv2.imdecode(np.frombuffer(data, np.uint8), 1), cv2.COLOR_BGR2RGB)); ax[0].set_title('Original')
    for k, c in enumerate((None, 0.10)):
        res = predict_urban_damage(data, conf=c, return_images=True)
        ann = cv2.imdecode(np.frombuffer(res.pop('annotated_png'), np.uint8), 1); res.pop('original_png')
        ax[k + 1].imshow(cv2.cvtColor(ann, cv2.COLOR_BGR2RGB))
        ax[k + 1].set_title(f"conf={res['conf_used']:.2f}: {res['num_defects']} defeito(s) {res['counts']} | {res['inference_ms']} ms", fontsize=9)
        print(name, json.dumps({k2: res[k2] for k2 in ('num_defects', 'counts', 'detections', 'inference_ms', 'processing_time_ms', 'device', 'imgsz')}, ensure_ascii=False))
    for a in ax: a.axis('off')
    plt.tight_layout(); plt.show()

## 10) Relatório com os números reais

In [ ]:
sh('python make_report.py')
from IPython.display import Markdown, display
display(Markdown(open('reports/RESULTADOS.md', encoding='utf-8').read()))

## 11) 🖥️ Interface CitizenTech (Gradio) — use para gravar o vídeo

In [ ]:
from app_gradio import build_app
build_app().launch(share=True, debug=False)

## 12) ✅ Checklist automático

In [ ]:
def ok(p): return '✅' if (glob.glob(p) if '*' in p else os.path.exists(p)) else '❌'
m = json.load(open('reports/metrics.json')) if os.path.exists('reports/metrics.json') else {}
rows = [('Base RDD2022 usada', 'data/extra/rdd2022/source.json'), ('Split sem data leakage (relatório)', 'reports/split_report.json'),
        ('Pesos treinados (best.pt)', 'weights/best.pt'), ('Precision/Recall/F1/mAP (metrics.json)', 'reports/metrics.json'),
        ('Matriz de confusão', 'reports/confusion_matrix.png'), ('Curvas de treino', f'{BACKUP_DIR}/training_curves.png'),
        ('Latência/FPS (T4/CPU)', 'reports/metrics.json'), ('Justificativa do limiar (RESULTADOS.md)', 'reports/RESULTADOS.md'),
        ('Análise de erros', 'reports/error_analysis/*.jpg'), ('Demonstrações (mosaicos/painel)', 'reports/demo/dashboard.png'),
        ('Fotos externas processadas', 'results/external/summary.csv'), ('Sequência de imagens', 'results/sequence/sequence_summary.json'),
        ('Interface (app_gradio.py)', 'app_gradio.py'), ('Fotos SEM danos (falso alarme)', 'tests/external_images/sem_danos/*.jp*g')]
for t, p in rows: print(ok(p), t)
if m.get('benchmark'): print('\nLatência medida em:', [b['dispositivo'] + f" @ {b['imgsz']}px" for b in m['benchmark'].values()])
print('\nPendente manual: "Observações visuais" em reports/RESULTADOS.md (sombras, texturas, iluminação, obstáculos).')

## 13) Baixar tudo (+ cópia no Drive)
💡 Antes de salvar/compartilhar o notebook: **Editar → Limpar todas as saídas**.

In [ ]:
sh('zip -qr /content/resultados_cv.zip weights/*.pt weights/threshold.json reports results minhas_fotos runs/eval_test')
shutil.copy('/content/resultados_cv.zip', BACKUP_DIR)
from google.colab import files
files.download('/content/resultados_cv.zip')